# Notebook 19 — Horse and Pedigree Identity

## Bounded question

> What do the runner-level `horse`, `sire`, `dam` and `damsire` fields represent in the source, how stable and complete are their labels, and which identity or pedigree relationships can be preserved safely without inventing entity equivalence from names alone?

## Initial governed scope

This notebook investigates four runner-level source-text fields:

- `horse`
- `sire`
- `dam`
- `damsire`

The source-field governance register assigns all four to the `horse_and_pedigree_identity` family, requires their raw values to be preserved and leaves their semantics pending.

The investigation begins with source lineage and physical profiling only. At this stage, no assumption is made that:

- a source string is a stable real-world entity identifier;
- identical strings always refer to the same horse;
- different strings always refer to different horses;
- a terminal country suffix is authoritative nationality evidence;
- stripping a suffix creates a globally unique name;
- `horse + country suffix` is a permanent natural key;
- repeated pedigree labels are complete, correct or internally consistent.

Raw source labels, parsed display names, embedded suffixes, source-level label identity, provisional entity candidates, verified real-world entities and pedigree assertions will remain separate concepts. Any future normalization must be reversible and must preserve physical source lineage, confidence and review status.

## Stage 1 — Source lineage and governed population

This stage establishes the immutable source, read-only controls, complete governed runner population and provisional race key before interpreting any horse or pedigree label.

The source is:

- database: `data/raw/form_2015-present/form_2015-present/raceform.db`
- table: `data`
- governed row predicate: `rowid <> 1`
- provisional race identity: `date + course + off`

The established source population is expected to contain:

- 1,851,285 governed runner rows;
- 189,043 provisional races;
- 37 source columns.

The first code cell opens SQLite in read-only mode, confirms the source schema, reconciles the governed runner and provisional-race counts, and confirms that `horse`, `sire`, `dam` and `damsire` are present. It does not parse, normalize or interpret any name.


In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd


# Resolve the immutable source explicitly from the notebook directory.
PROJECT_ROOT = Path.cwd().resolve().parent
SOURCE_DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"
RACE_KEY_COLUMNS = ["date", "course", "off"]
HORSE_IDENTITY_FIELDS = ["horse", "sire", "dam", "damsire"]

EXPECTED_RUNNER_ROWS = 1_851_285
EXPECTED_PROVISIONAL_RACES = 189_043
EXPECTED_SOURCE_COLUMNS = 37

if not SOURCE_DB_PATH.exists():
    raise FileNotFoundError(f"Source database not found: {SOURCE_DB_PATH}")

# Open SQLite read-only so the notebook cannot mutate the source database.
connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    schema = pd.read_sql_query(f"PRAGMA table_info({SOURCE_TABLE})", connection)
    source_columns = schema["name"].tolist()

    missing_fields = [
        field
        for field in HORSE_IDENTITY_FIELDS
        if field not in source_columns
    ]
    if missing_fields:
        raise AssertionError(f"Missing horse-identity fields: {missing_fields}")

    runner_rows = connection.execute(
        f"SELECT COUNT(*) FROM {SOURCE_TABLE} WHERE {DATA_ROW_PREDICATE}"
    ).fetchone()[0]

    provisional_races = connection.execute(
        f"""
        SELECT COUNT(*)
        FROM (
            SELECT DISTINCT date, course, off
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
        )
        """
    ).fetchone()[0]
finally:
    connection.close()

assert runner_rows == EXPECTED_RUNNER_ROWS
assert provisional_races == EXPECTED_PROVISIONAL_RACES
assert len(source_columns) == EXPECTED_SOURCE_COLUMNS

source_lineage_summary = pd.DataFrame(
    [
        ("source database", SOURCE_DB_PATH.relative_to(PROJECT_ROOT).as_posix()),
        ("source table", SOURCE_TABLE),
        ("data-row predicate", DATA_ROW_PREDICATE),
        ("runner rows", runner_rows),
        ("provisional races", provisional_races),
        ("source columns", len(source_columns)),
        ("provisional race key", " + ".join(RACE_KEY_COLUMNS)),
        ("horse-identity fields present", ", ".join(HORSE_IDENTITY_FIELDS)),
    ],
    columns=["measure", "value"],
)

print("Governed source population confirmed")
source_lineage_summary


## Stage 2 — Confirm inherited source-field governance

Before profiling the contents of the four fields, this stage reads their existing rows from `data/reference/source_field_governance.csv`.

The check is limited to confirming the inherited starting position:

- each field is recorded at runner grain;
- each belongs to `horse_and_pedigree_identity`;
- raw preservation is required;
- the current blank policy is retained exactly as governed;
- semantic status remains pending;
- the existing governing notebook attribution is preserved.

This stage does not revise the register or infer anything from the field labels themselves.


In [ ]:
SOURCE_FIELD_GOVERNANCE_PATH = (
    PROJECT_ROOT / "data" / "reference" / "source_field_governance.csv"
)

if not SOURCE_FIELD_GOVERNANCE_PATH.exists():
    raise FileNotFoundError(
        f"Source-field governance register not found: "
        f"{SOURCE_FIELD_GOVERNANCE_PATH}"
    )

source_field_governance = pd.read_csv(SOURCE_FIELD_GOVERNANCE_PATH)

horse_identity_governance = (
    source_field_governance.loc[
        source_field_governance["source_field"].isin(HORSE_IDENTITY_FIELDS),
        [
            "ordinal",
            "source_field",
            "declared_type",
            "grain",
            "field_family",
            "raw_preservation",
            "blank_policy",
            "dash_policy",
            "zero_policy",
            "governed_by",
            "status",
        ],
    ]
    .sort_values("ordinal")
    .reset_index(drop=True)
)

assert horse_identity_governance["source_field"].tolist() == HORSE_IDENTITY_FIELDS
assert horse_identity_governance["grain"].eq("runner").all()
assert horse_identity_governance["field_family"].eq(
    "horse_and_pedigree_identity"
).all()
assert horse_identity_governance["raw_preservation"].eq("required").all()
assert horse_identity_governance["status"].eq("pending_semantics").all()

print("Inherited horse-identity governance confirmed")
horse_identity_governance
